# 02 - Audio EDA

Durations, sample rates, channels, one example waveform/spectrogram and a first look at acoustic cues vs. the label.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
from src.config import load_config, repo_dir
from src.pipeline import get_context

# Dataset location: leave None to use the SHL_DATA_ROOT environment variable (or config.data_root),
# or set it here, e.g. DATA_ROOT = r"C:\Users\USERNAME\path\Dataset_Final"
DATA_ROOT = None
try:
    ctx = get_context(DATA_ROOT); cfg = ctx.cfg
    print("Data root:", ctx.paths.root)
except Exception as exc:
    ctx, cfg = None, load_config()
    print("Data root NOT configured ->", exc)
PENDING = "Results will be populated after local execution."
TABLES = repo_dir(cfg, "reports_dir") / "tables"
LOG_PATH = repo_dir(cfg, "experiments_dir") / "experiment_log.csv"
def have(*paths): return all(Path(p).exists() for p in paths)

In [ ]:
have_audio = ctx is not None and ctx.paths.train_audio_dir.is_dir()
if not have_audio:
    print("Train audio folder not found.", PENDING)
else:
    from src.audio.quality import probe_audio
    from src.data.loader import load_all_metadata
    meta = load_all_metadata(ctx.paths)
    inv_path = TABLES / "audio_inventory.csv"
    if inv_path.exists():
        inv = pd.read_csv(inv_path)
    else:
        from tqdm.auto import tqdm
        inv = pd.DataFrame([probe_audio(ctx.paths.train_audio_dir / f) | {"split": "train"} for f in tqdm(meta.train.filename)])
    inv = inv[inv["split"] == "train"] if "split" in inv else inv
    display(inv.describe(include="all").T)

## Duration, sample-rate and channel distributions

In [ ]:
if have_audio:
    from src.reporting import plot_hist, plot_value_counts
    ok = inv[inv["ok"]]
    plot_hist(ok["duration"], "Audio duration (train)", "Seconds", name="duration_distribution"); plt.show()
    plot_value_counts(ok["sample_rate"].astype(int), "Sample rate (train)", "Hz", name="sample_rate_distribution"); plt.show()
    print("channels:", ok["channels"].value_counts().to_dict())

## Example waveform and spectrogram (one low-score and one high-score file)

In [ ]:
if have_audio:
    from src.audio.preprocessing import load_audio
    from src.reporting import plot_waveform_spectrogram
    tr = meta.train.sort_values("label")
    for tag, row in (("lowest label", tr.iloc[0]), ("highest label", tr.iloc[-1])):
        y_, sr_ = load_audio(ctx.paths.train_audio_dir / row.filename, sr=cfg["audio"]["sample_rate"])
        plot_waveform_spectrogram(y_, sr_, title=f"{row.filename} (label {row.label}, {tag})", name=f"waveform_spectrogram_{tag.split()[0]}")
        plt.show()

## Acoustic cues vs label (requires `scripts/extract_features.py` to have run)

In [ ]:
f = ctx.processed("audio_features") / "train_audio_features.csv" if ctx else None
if f is not None and f.exists():
    af = pd.read_csv(f).merge(meta.train, on="filename")
    af = af[af["status"] == "ok"]
    print(f"{len(af)} / {len(meta.train)} files with acoustic features")
    cols = ["aud_duration", "aud_speech_ratio", "aud_silence_ratio", "aud_pauses_per_min", "aud_pause_mean", "aud_onset_rate", "aud_pitch_std"]
    fig, axes = plt.subplots(2, 4, figsize=(16, 6))
    for ax, c in zip(axes.ravel(), cols):
        ax.scatter(af["label"] + np.random.normal(0, .04, len(af)), af[c], s=8, alpha=.4); ax.set(title=c, xlabel="label")
    axes.ravel()[-1].axis("off"); fig.tight_layout(); plt.show()
    display(af[cols + ["label"]].corr(method="spearman")["label"].to_frame("spearman vs label"))
else:
    print(PENDING)